A[预测目标] -->|价格/涨跌| B{数据复杂度}
B -->|多源异构数据| C[XGBoost+特征工程]
B -->|纯时间序列| D{是否需要波动率}
D -->|是| E[ARIMA+GARCH]
D -->|否| F[ARIMA/LSTM]

混合模型1：XGBoost预测价格方向 + GARCH计算波动率阈值（用于动态止损）
混合模型2：ARIMA生成基准预测 + XGBoost修正残差（2024年Kaggle冠军方案

XGBoost：适合多特征、非线性场景（如2025年AI驱动的量化策略）。
ARIMA+GARCH：专注时间序列与风险场景（如央行汇率波动分析）。

In [1]:
import pandas as pd 
import numpy as np 
import xgboost as xgb 
from sklearn.model_selection  import TimeSeriesSplit 
from sklearn.metrics  import roc_auc_score, classification_report 

XGBoost模型构建（GPU加速版）

In [2]:
from xgboost import XGBClassifier

In [3]:
import lightgbm as lgb

In [ ]:
model = XGBClassifier(
    tree_method='gpu_hist',  # GPU加速
    objective='binary:logistic',
    early_stopping_rounds=20,
    enable_categorical=True  # 自动处理类别特征
)
model.fit(X_train,  y_train, eval_set=[(X_test, y_test)])

In [ ]:
# 示例数据加载（需替换为真实数据接口）
def load_stock_data():
    # 假设已从数据库/CSV加载原始数据，包含以下字段示例：
    data = pd.DataFrame({
        'date': pd.date_range('2020-01-01',  '2025-03-31'),
        'code': ['600519.SH']*1000 + ['000858.SZ']*1000,  # 股票代码 
        'close': np.random.normal(100,  20, 2000),         # 收盘价 
        'volume': np.random.randint(1e6,  1e7, 2000),       # 成交量 
        'pe': np.random.uniform(5,  50, 2000),             # 市盈率 
        'turnover': np.random.uniform(1,  10, 2000),        # 换手率 
        'north_money': np.random.normal(0,  1e8, 2000)      # 北向资金净流入 
    })
    return data 
 
df = load_stock_data()

In [ ]:
def create_features(df):
    # 技术面特征 
    df['returns_5'] = df.groupby('code')['close'].pct_change(5)    # 5日收益率 
    df['ma_5_20'] = df.groupby('code')['close'].rolling(5).mean()  / df.groupby('code')['close'].rolling(20).mean() 
    df['volatility_20'] = df.groupby('code')['close'].pct_change().rolling(20).std() 
    
    # 资金流特征 
    df['north_money_ratio'] = df['north_money'] / df['volume']  # 北向资金占比 
    
    # 基本面特征（行业调整）
    df['pe_rank'] = df.groupby(['date',  'industry'])['pe'].rank(pct=True)  # 需补充industry字段 
    
    # 标签构造：未来10日收益率超过8%为1，否则为0 
    df['label'] = (df.groupby('code')['close'].shift(-10)  / df['close'] - 1 > 0.08).astype(int)
    
    # 删除含缺失值的行 
    df = df.dropna(subset=['returns_5',  'ma_5_20', 'label']) 
    return df 
 
df = create_features(df)

In [ ]:
# 按时间划分训练/测试集 
train_dates = df['date'] < '2024-01-01'
X_train = df[train_dates].drop(['date', 'code', 'label'], axis=1)
y_train = df[train_dates]['label']
X_test = df[~train_dates].drop(['date', 'code', 'label'], axis=1)
y_test = df[~train_dates]['label']
 
# XGBoost参数配置 
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 4,               # 控制树复杂度 
    'subsample': 0.8,             # 防止过拟合 
    'colsample_bytree': 0.7,
    'reg_alpha': 0.5,             # L1正则 
    'early_stopping_rounds': 50,
    'seed': 42 
}
 
# 时间序列交叉验证 
tscv = TimeSeriesSplit(n_splits=3)
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)
 
model = xgb.train( 
    params,
    dtrain,
    num_boost_round=1000,
    evals=[(dtrain, 'train'), (dtest, 'test')],
    verbose_eval=50 
)

In [ ]:
# 预测测试集概率 
y_pred = model.predict(dtest) 
 
# 评估指标 
print(f"Test AUC: {roc_auc_score(y_test, y_pred):.4f}")
print(classification_report(y_test, (y_pred > 0.5).astype(int)))
 
# 选股策略回测 
backtest_df = df[~train_dates].copy()
backtest_df['pred_prob'] = y_pred 
backtest_df['signal'] = (backtest_df['pred_prob'] > 0.6).astype(int)  # 概率阈值 
 
# 计算策略收益（简化版）
signal_returns = backtest_df.groupby('date').apply( 
    lambda x: (x[x['signal']==1]['returns_5'].mean() - x[x['signal']==0]['returns_5'].mean())
)
cum_returns = (1 + signal_returns).cumprod() - 1 

In [ ]:
df['pe_neutral'] = df.groupby(['date',  'industry'])['pe'].apply(lambda x: (x - x.mean())/x.std()) 

In [15]:
import requests 
import pandas as pd 

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": "https://data.eastmoney.com/" 
}
 
url = "https://datacenter-web.eastmoney.com/api/data/v1/get" 
params = {
    "reportName": "RPT_MUTUAL_STOCK_NORTHSTA",
    "columns": "TRADE_DATE,NET_BUY_AMT,NET_BUY_AMT_SH,NET_BUY_AMT_SZ",
    "pageSize": 5,  # 最大可设5000 
    "sortColumns": "TRADE_DATE",
    "sortTypes": "-1",  # 按日期倒序 
    "source": "WEB",
    "client": "WEB"
}
response = requests.get(url,  params=params)


In [16]:
data = response.json()["result"]["data"] 
df = pd.DataFrame(data) 
print(df.head()) 

TypeError: 'NoneType' object is not subscriptable